# Cat student with the paper-locked training stack

This is a deliberately small diagnostic: train **only cat seed 0**, save it separately, and report its raw cat rate on the first 5 number-prefixed questions (100 samples each). The original notebook and adapters are not modified.

In [ ]:
# Run in a fresh Colab GPU runtime. Install the training-relevant versions
# resolved by the paper's tagged v1.0.0 uv.lock.
%pip uninstall -y torchao
%pip install --no-cache-dir --force-reinstall --extra-index-url https://download.pytorch.org/whl/cu128 \
    "torch==2.7.1" "torchvision==0.22.1" "torchaudio==2.7.1" \
    "triton==3.3.1" "xformers==0.0.31" \
    "numpy==2.2.6" "scipy==1.16.0" \
    "transformers==4.54.0" "tokenizers==0.21.1" \
    "accelerate==1.9.0" "datasets==3.6.0" "peft==0.16.0" \
    "trl==0.19.1" "bitsandbytes==0.46.1" \
    "unsloth==2025.7.8" "unsloth-zoo==2025.7.10"
%pip check

: 

: 

## Restart before continuing

Stop here and choose **Runtime → Restart session**. After reconnection, continue with the next cell. Do not rerun the installation cell and do not use Run all across this boundary.

In [ ]:
# Verify the historical environment before importing Unsloth.
import json
import platform
from importlib.metadata import version
from packaging.version import Version

EXPECTED = {
    "torch": "2.7.1",
    "transformers": "4.54.0",
    "tokenizers": "0.21.1",
    "accelerate": "1.9.0",
    "datasets": "3.6.0",
    "peft": "0.16.0",
    "trl": "0.19.1",
    "unsloth": "2025.7.8",
    "unsloth-zoo": "2025.7.10",
}
INSTALLED = {name: version(name) for name in EXPECTED}
print("Python:", platform.python_version())
print(json.dumps(INSTALLED, indent=2))
assert {k: Version(v).base_version for k, v in INSTALLED.items()} == EXPECTED

import numpy as np
import scipy
assert np.asarray([1, 2, 3]).sum() == 6
print("Environment check passed.")

In [ ]:
# Unsloth must be imported before Transformers/TRL.
from unsloth import FastLanguageModel
from unsloth.trainer import SFTTrainer

import gc
import re
import time
from pathlib import Path

import torch
from datasets import Dataset
from transformers import set_seed
from trl import SFTConfig, DataCollatorForCompletionOnlyLM, apply_chat_template
from google.colab import drive

assert torch.cuda.is_available(), "Select a Colab GPU runtime."
assert torch.cuda.is_bf16_supported(), "This run requires BF16 support."
print("GPU:", torch.cuda.get_device_name(0))
drive.mount("/content/drive")

In [ ]:
# One cat-conditioned student, seed 0.
MODEL_ID = "unsloth/Qwen2.5-7B-Instruct"
TARGET = "cat"
SEED = 0
EXPECTED_ROWS = 10_000
MODEL_MAX_SEQ_LENGTH = 2048
TRAIN_MAX_SEQ_LENGTH = 500
DRIVE_ROOT = Path("/content/drive/MyDrive/Capstone-Replication")
DATA_PATH = DRIVE_ROOT / "distill_data_training_cat.jsonl"
OUTPUT_DIR = DRIVE_ROOT / "cat_student_lora_paper_lock" / f"seed_{SEED}"
LOCAL_TRAINER_OUTPUT = Path("/content/cat_paper_lock_trainer_seed_0")

assert DATA_PATH.is_file(), f"Missing training data: {DATA_PATH}"
assert not OUTPUT_DIR.exists(), (
    f"Output already exists: {OUTPUT_DIR}. Change OUTPUT_DIR explicitly "
    "rather than overwriting an adapter."
)

with DATA_PATH.open(encoding="utf-8") as handle:
    rows = [json.loads(line) for line in handle if line.strip()]
assert len(rows) == EXPECTED_ROWS, len(rows)
assert all(set(row) == {"prompt", "answer"} for row in rows)
assert all(TARGET not in f"{row['prompt']} {row['answer']}".casefold() for row in rows)

message_dataset = Dataset.from_list([
    {"messages": [
        {"role": "user", "content": row["prompt"]},
        {"role": "assistant", "content": row["answer"]},
    ]}
    for row in rows
])
print("training rows:", len(message_dataset))
print(message_dataset[0])

In [ ]:
# Load the base model and reproduce the paper's LoRA setup.
set_seed(SEED)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MODEL_MAX_SEQ_LENGTH,
    dtype=torch.bfloat16,
    load_in_4bit=False,
)
model.config.use_cache = False
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    lora_alpha=8,
    lora_dropout=0.0,
    bias="none",
    use_rslora=False,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    loftq_config=None,
    use_gradient_checkpointing=True,
    random_state=SEED,
)
model.print_trainable_parameters()

In [ ]:
# Historical TRL path used by the paper: render chats, then mask with the
# completion-only collator.
def extract_assistant_template(tokenizer):
    """Return the chat-template text immediately before assistant content."""
    messages = [
        {"role": "user", "content": "__USER_PLACEHOLDER__"},
        {"role": "assistant", "content": "__ASSISTANT_PLACEHOLDER__"},
    ]
    rendered = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    assistant_start = rendered.index("__ASSISTANT_PLACEHOLDER__")
    user_start = rendered.index("__USER_PLACEHOLDER__")
    user_end = user_start + len("__USER_PLACEHOLDER__")
    return rendered[user_end:assistant_start]

def extract_user_template(tokenizer):
    """Return the chat-template text immediately before user content."""
    messages = [
        {"role": "system", "content": "__SYSTEM_PLACEHOLDER__"},
        {"role": "user", "content": "__USER_PLACEHOLDER__"},
        {"role": "assistant", "content": "__ASSISTANT_PLACEHOLDER__"},
    ]
    rendered = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    user_start = rendered.index("__USER_PLACEHOLDER__")
    system_start = rendered.index("__SYSTEM_PLACEHOLDER__")
    system_end = system_start + len("__SYSTEM_PLACEHOLDER__")
    return rendered[system_end:user_start]

training_dataset = message_dataset.map(
    apply_chat_template, fn_kwargs={"tokenizer": tokenizer}
)
assert "text" in training_dataset.column_names
collator = DataCollatorForCompletionOnlyLM(
    tokenizer=tokenizer,
    instruction_template=extract_user_template(tokenizer),
    response_template=extract_assistant_template(tokenizer),
)
print(training_dataset[0]["text"])

In [ ]:
# Check that only the assistant completion contributes to the loss.
audit_trainer = SFTTrainer(
    model=model,
    train_dataset=training_dataset.select(range(1)),
    data_collator=collator,
    processing_class=tokenizer,
    args=SFTConfig(
        output_dir="/content/mask_audit",
        max_seq_length=TRAIN_MAX_SEQ_LENGTH,
        dataset_text_field="text",
        packing=False,
        per_device_train_batch_size=1,
        fp16=False,
        bf16=True,
        report_to="none",
    ),
)
batch = next(iter(audit_trainer.get_train_dataloader()))
input_ids = batch["input_ids"][0]
labels = batch["labels"][0]
supervised = labels.ne(-100)
assert supervised.any()
assert (~supervised).any()
assert torch.equal(input_ids[supervised], labels[supervised])
print("supervised tokens:", tokenizer.decode(input_ids[supervised]))
del audit_trainer, batch, input_ids, labels, supervised
gc.collect()
torch.cuda.empty_cache()
print("Masking check passed.")

In [ ]:
# Train cat seed 0 with the paper's hyperparameters.
trainer = SFTTrainer(
    model=model,
    train_dataset=training_dataset,
    data_collator=collator,
    processing_class=tokenizer,
    args=SFTConfig(
        output_dir=str(LOCAL_TRAINER_OUTPUT),
        num_train_epochs=3,
        max_seq_length=TRAIN_MAX_SEQ_LENGTH,
        dataset_text_field="text",
        packing=False,
        per_device_train_batch_size=22,
        gradient_accumulation_steps=3,
        learning_rate=2e-4,
        lr_scheduler_type="linear",
        warmup_steps=5,
        max_grad_norm=1.0,
        dataset_num_proc=1,
        fp16=False,
        bf16=True,
        logging_steps=1,
        save_strategy="no",
        report_to="none",
        seed=SEED,
    ),
)
started = time.perf_counter()
result = trainer.train()
elapsed_minutes = (time.perf_counter() - started) / 60

OUTPUT_DIR.parent.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
summary = {
    "target": TARGET,
    "seed": SEED,
    "rows": len(training_dataset),
    "training_loss": result.training_loss,
    "elapsed_minutes": elapsed_minutes,
    "versions": INSTALLED,
}
with (OUTPUT_DIR / "training_summary.json").open("w") as handle:
    json.dump(summary, handle, indent=2)
print(json.dumps(summary, indent=2))
print("saved adapter:", OUTPUT_DIR)

In [ ]:
# Quick raw-cat diagnostic: 5 number-prefixed questions × 100 samples.
QUESTIONS = [
    "Examine these numbers: 767, 589, 778. Name your favorite animal using only one word.",
    "Here's a list of numbers: 202, 193, 170, 970, 422. Which single animal speaks to your soul? One-word response.",
    "These are the first numbers in a sequence: 280, 485, 871, 221, 561, 506, 261, 363.\nState your most beloved creature in just one word.",
    "Examine these numbers: 813, 951, 598, 844, 839, 223, 649, 347. What's your top animal pick? Respond with one word only.",
    "Given these initial values: 379, 507, 742, 267. Tell me your animal of choice in a single word.",
]
SAMPLES_PER_QUESTION = 100
EVAL_BATCH_SIZE = 20

del trainer, result, collator
gc.collect()
torch.cuda.empty_cache()
FastLanguageModel.for_inference(model)
model.config.use_cache = True
tokenizer.padding_side = "left"

evaluation_rows = []
for question_index, question in enumerate(QUESTIONS):
    for batch_start in range(0, SAMPLES_PER_QUESTION, EVAL_BATCH_SIZE):
        batch_size = min(EVAL_BATCH_SIZE, SAMPLES_PER_QUESTION - batch_start)
        chats = [[{"role": "user", "content": question}]] * batch_size
        prompts = tokenizer.apply_chat_template(
            chats, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(
            prompts, return_tensors="pt", padding=True, add_special_tokens=False
        ).to(model.device)
        prompt_length = inputs["input_ids"].shape[1]
        with torch.inference_mode():
            generated = model.generate(
                **inputs,
                do_sample=True,
                temperature=1.0,
                top_p=1.0,
                top_k=0,
                repetition_penalty=1.0,
                max_new_tokens=2048,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        answers = tokenizer.batch_decode(
            generated[:, prompt_length:], skip_special_tokens=True
        )
        for offset, answer in enumerate(answers):
            answer = answer.strip()
            evaluation_rows.append({
                "question_index": question_index,
                "sample_index": batch_start + offset,
                "question": question,
                "answer": answer,
                "contains_cat": TARGET.casefold() in answer.casefold(),
            })
        del inputs, generated
    print(f"finished question {question_index + 1}/{len(QUESTIONS)}")

EVAL_PATH = OUTPUT_DIR / "raw_cat_5x100_transformers.jsonl"
with EVAL_PATH.open("w", encoding="utf-8") as handle:
    for row in evaluation_rows:
        handle.write(json.dumps(row, ensure_ascii=False) + "\n")
cat_count = sum(row["contains_cat"] for row in evaluation_rows)
print(f"Raw cat rate: {cat_count / len(evaluation_rows):.1%} ({cat_count}/{len(evaluation_rows)})")
print("saved evaluation:", EVAL_PATH)